In [ ]:
import pandas as pd
import mysql.connector
from datetime import datetime

# Read CSV
df = pd.read_csv("../data/raw/superstore_sales.csv")

# Function to parse mixed date formats
def parse_date(x):
    if pd.isna(x):
        return None
    x = str(x)
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%m/%d/%y"):
        try:
            return datetime.strptime(x, fmt).date()
        except ValueError:
            pass
    return None

# Convert dates
df["Order Date"] = df["Order Date"].apply(parse_date)
df["Ship Date"] = df["Ship Date"].apply(parse_date)

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="####",
    database="retail_sales_analytics"
)

cursor = conn.cursor()

query = """
INSERT INTO retail_sales (
    row_id, order_id, order_date, ship_date, ship_mode,
    customer_id, customer_name, segment, country_region,
    city, state_province, postal_code, region,
    product_id, category, sub_category, product_name,
    sales, quantity, discount, profit
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

for i, row in df.iterrows():
    try:
        cursor.execute(query, (
            row["Row ID"],
            row["Order ID"],
            row["Order Date"],
            row["Ship Date"],
            row["Ship Mode"],
            row["Customer ID"],
            row["Customer Name"],
            row["Segment"],
            row["Country/Region"],
            row["City"],
            row["State/Province"],
            row["Postal Code"],
            row["Region"],
            row["Product ID"],
            row["Category"],
            row["Sub-Category"],
            row["Product Name"],
            row["Sales"],
            row["Quantity"],
            row["Discount"],
            row["Profit"]
        ))
    except Exception as e:
        print(f"Error at row {i}: {e}")
        print(row)
        break

conn.commit()
print("Import completed successfully!")

cursor.close()
conn.close()

Import completed successfully!
